In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from students.romanova.lesson2 import Exercise as LogisticModel


def apply_normalization(training_data, validation_data, test_data):
    avg = training_data.mean(axis=0)
    deviation = training_data.std(axis=0)
    deviation[deviation == 0] = 1.0

    training_normalized = (training_data - avg) / deviation
    validation_normalized = (validation_data - avg) / deviation
    test_normalized = (test_data - avg) / deviation

    return training_normalized, validation_normalized, test_normalized


def run_experiment(step, group_size, X_tr, X_v, y_tr, y_v, max_passes, random_state):
    generator = np.random.default_rng(random_state)
    binary_classifier = LogisticModel.create_logistic_model(num_features=X_tr.shape[1], rng=generator)

    LogisticModel.fit(binary_classifier, X_tr, y_tr, lr=step, n_iter=max_passes, batch_size=group_size)

    validation_auroc = binary_classifier.metric(X_v, y_v, metric="auroc")
    return validation_auroc, binary_classifier


def main():
    iris_dataset = load_iris()
    feature_matrix = iris_dataset.data
    target_vector = iris_dataset.target

    binary_labels = (target_vector < 1).astype(int)

    X_interim, X_test_final, y_interim, y_test_final = train_test_split(
        feature_matrix, binary_labels, test_size=0.4, random_state=42, stratify=binary_labels
    )

    X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
        X_interim, y_interim, test_size=0.5, random_state=42, stratify=y_interim
    )

    X_train_norm, X_val_norm, X_test_norm = apply_normalization(X_train_final, X_val_final, X_test_final)

    epoh = 25
    random_seeds = [42, 123, 456, 2026]

    learning_rates = [0.001, 0.003, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]
    batch_sizes = [2, 4, 8, 16, 32, None]

    optimal_settings = {"lr": 0, "batch_size": None, "auroc_mean": 0.0, "auroc_std": 1.0}

    for current_lr in learning_rates:
        for current_bs in batch_sizes:
            performance_history = []

            for current_seed in random_seeds:
                auroc_result, _ = run_experiment(
                    current_lr, current_bs, X_train_norm, X_val_norm, y_train_final, y_val_final, epoh, current_seed
                )
                performance_history.append(auroc_result)

            average_performance = np.mean(performance_history)
            performance_spread = np.std(performance_history)

            improved_mean = average_performance > optimal_settings["auroc_mean"]
            equal_mean_less_variance = (
                average_performance == optimal_settings["auroc_mean"]
                and performance_spread < optimal_settings["auroc_std"]
            )

            if improved_mean or equal_mean_less_variance:
                optimal_settings.update(
                    {
                        "lr": current_lr,
                        "batch_size": current_bs,
                        "auroc_mean": average_performance,
                        "auroc_std": performance_spread,
                    }
                )

    print(f"lr: {optimal_settings['lr']}")
    print(f"batch size: {optimal_settings['batch_size']}")
    print(f"auroc value: {optimal_settings['auroc_mean']:.4f} +- {optimal_settings['auroc_std']:.4f}")
    print(f"epoh: {epoh}")

    return optimal_settings


if __name__ == "__main__":
    main()